In [ ]:
import socket
import ssl
import threading
import time
import os
from datetime import datetime, timedelta, UTC

HOST = '127.0.0.1'
PORT = 8443

CERT_FILE = "tls_test_server.crt"
KEY_FILE = "tls_test_server.key"


def generate_self_signed_cert():
    """Generate self-signed certificate using cryptography"""
    try:
        from cryptography import x509
        from cryptography.x509.oid import NameOID
        from cryptography.hazmat.primitives import hashes
        from cryptography.hazmat.primitives.asymmetric import rsa
        from cryptography.hazmat.primitives import serialization

        print("🔧 Generating self-signed certificate...")

        private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)

        subject = issuer = x509.Name([
            x509.NameAttribute(NameOID.COUNTRY_NAME, "US"),
            x509.NameAttribute(NameOID.STATE_OR_PROVINCE_NAME, "California"),
            x509.NameAttribute(NameOID.LOCALITY_NAME, "San Jose"),
            x509.NameAttribute(NameOID.ORGANIZATION_NAME, "TLS Demo"),
            x509.NameAttribute(NameOID.COMMON_NAME, "localhost"),
        ])

        now = datetime.now(UTC)
        
        cert = (
            x509.CertificateBuilder()
            .subject_name(subject)
            .issuer_name(issuer)
            .public_key(private_key.public_key())
            .serial_number(x509.random_serial_number())
            .not_valid_before(now)
            .not_valid_after(now + timedelta(days=365))
            .add_extension(
                x509.SubjectAlternativeName([x509.DNSName("localhost")]),
                critical=False,
            )
            .sign(private_key, hashes.SHA256())
        )

        with open(CERT_FILE, "wb") as f:
            f.write(cert.public_bytes(serialization.Encoding.PEM))

        with open(KEY_FILE, "wb") as f:
            f.write(private_key.private_bytes(
                encoding=serialization.Encoding.PEM,
                format=serialization.PrivateFormat.TraditionalOpenSSL,
                encryption_algorithm=serialization.NoEncryption(),
            ))

        print("✅ Self-signed certificate generated successfully.")
        return True

    except ImportError:
        print("❌ 'cryptography' library not found. Install with: pip install cryptography")
        return False
    except Exception as e:
        print(f"❌ Certificate generation failed: {e}")
        return False


# ====================== TLS SERVER ======================
def run_server():
    if not (os.path.exists(CERT_FILE) and os.path.exists(KEY_FILE)):
        if not generate_self_signed_cert():
            print("Cannot start server without certificate.")
            return

    context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
    context.load_cert_chain(certfile=CERT_FILE, keyfile=KEY_FILE)

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind((HOST, PORT))
        sock.listen(5)
        print(f"🔒 TLS Server listening on https://{HOST}:{PORT}")

        with context.wrap_socket(sock, server_side=True) as ssock:
            while True:
                try:
                    conn, addr = ssock.accept()
                    print(f"✅ Client connected from {addr}")
                    data = conn.recv(1024)
                    if data:
                        print(f"📨 Received: {data.decode('utf-8', errors='ignore')}")
                        # Fixed: Use .encode() instead of bytes literal with emoji
                        response = "✅ TLS Server received your message!".encode('utf-8')
                        conn.sendall(response)
                    conn.close()
                except Exception as e:
                    print(f"Connection error: {e}")


# ====================== TLS CLIENT ======================
def run_client():
    time.sleep(2)  # Give server time to start

    context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
    context.load_verify_locations(CERT_FILE)
    context.check_hostname = False  # localhost demo

    try:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            with context.wrap_socket(sock, server_hostname="localhost") as ssock:
                ssock.connect((HOST, PORT))
                print("🔐 Secure TLS connection established!")

                message = "I have been very jealous for the LORD God of hosts: for the children of Israel have forsaken thy covenant, thrown down thine altars, and slain thy prophets with the sword; and I, even I only, am left; and they seek my life, to take it away.".encode('utf-8')
                ssock.sendall(message)

                response = ssock.recv(1024)
                print(f"📨 Server replied: {response.decode('utf-8')}")
    except Exception as e:
        print(f"Client error: {e}")


# ====================== MAIN ======================
if __name__ == "__main__":
    print("🚀 TLS Demo (Server + Client)\n")

    server_thread = threading.Thread(target=run_server, daemon=True)
    server_thread.start()

    run_client()

    print("\n✅ Demo completed. Press Ctrl+C to exit.")
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("\n\nShutting down...")

🚀 TLS Demo (Server + Client)

🔧 Generating self-signed certificate...
✅ Self-signed certificate generated successfully.
🔒 TLS Server listening on https://127.0.0.1:8443
🔐 Secure TLS connection established!
✅ Client connected from ('127.0.0.1', 50905)
📨 Received: I have been very jealous for the LORD God of hosts: for the children of Israel have forsaken thy covenant, thrown down thine altars, and slain thy prophets with the sword; and I, even I only, am left; and they seek my life, to take it away.
📨 Server replied: ✅ TLS Server received your message!

✅ Demo completed. Press Ctrl+C to exit.
